In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import glob
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
hicdir = f'{projdir}/snm3C/hicluster'
refdir = '/u/project/cluo/terencew/reference/hg38_igvf'
beddir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/bed'
bin_size = 25000

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

### 25kb bins and per-nucleus TopDom boundaries (as in ips_subtypes/ipynb/enrich/A01a_dmr_enrich_diff_boundaries)

In [ ]:
bins = pd.read_csv(f'{refdir}/bed/windows/25kb/GRCh38.25kb.bed', sep='\t', header=None)
bins.columns = ['chrom', 'start', 'end']
bins.index = [f'{x}_{y}' for x,y in zip(bins['chrom'], bins['start'])]
offsets = bins.reset_index().groupby('chrom', sort=False).apply(lambda x: x.index[0]).to_dict()
bins.shape

In [ ]:
domain_dir = f'{hicdir}/old_hicluster/imputed_matrix/25kb_resolution/merged/domains'
domain_paths = pd.Series({x.split('/')[-1].split('.')[0]: x for x in glob.glob(f'{domain_dir}/*.bed.gz')})
early_3c = early[early['assay'] == 'snm3C']
keep = early_3c.index[early_3c.index.isin(domain_paths.index)]
len(early_3c), len(keep)

In [ ]:
def load_boundary_bins(path):
    x = pd.read_csv(path, sep='\t', header=None)
    x = x[x[3] == 'boundary']
    idx = [offsets[c] + np.arange(s, e, bin_size) // bin_size for c, s, e in zip(x[0], x[1], x[2])]
    return np.unique(np.concatenate(idx)) if len(idx) > 0 else np.array([], dtype=int)

with ProcessPoolExecutor(max_workers=4) as executor:
    cell_bins = list(tqdm(executor.map(load_boundary_bins, domain_paths[keep].values), total=len(keep)))

In [ ]:
labels = early_3c.loc[keep, 'ips_subtype'].values
boundary_df = pd.DataFrame(index=bins.index)
non_boundary_df = pd.DataFrame(index=bins.index)
for s in ['IPS_main', 'IPS_alt']:
    tmp = [cell_bins[i] for i in np.where(labels == s)[0]]
    boundary_df[s] = np.bincount(np.concatenate(tmp), minlength=bins.shape[0])
    non_boundary_df[s] = len(tmp) - boundary_df[s]
pd.Series(labels).value_counts()

In [ ]:
mask = boundary_df.sum(axis=1) > 0
boundary_sum = boundary_df.loc[mask].values
non_boundary_sum = non_boundary_df.loc[mask].values
results = [chi2_contingency(np.array([boundary_sum[i], non_boundary_sum[i]]).T)[:2] for i in tqdm(range(boundary_sum.shape[0]))]
tested = pd.concat([boundary_df.loc[mask], non_boundary_df.loc[mask]], axis=1)
tested.columns = ['IPS_main_boundary', 'IPS_alt_boundary', 'IPS_main_domain', 'IPS_alt_domain']
tested['chi2'] = [x[0] for x in results]
tested['pval'] = [x[1] for x in results]
tested['padj'] = multipletests(tested['pval'], alpha=0.05, method='fdr_bh')[1]
tested['differential'] = tested['padj'] < 0.01
tested = bins.loc[tested.index].join(tested)
tested['size'] = tested['end'] - tested['start']
tested['differential'].sum(), tested.shape

### compare with published differential bins (all nuclei)

In [ ]:
pub = pd.read_csv('/u/project/cluo/terencew/claude/igvf/pgp/ips_subtypes/csv/enrich/ips_cluster_diff_bins.csv', sep='\t', index_col=0)
new = tested[tested['differential']]
len(pub), len(new), len(pub.index.intersection(new.index))

In [ ]:
new['IPS_main_frac'] = new['IPS_main_boundary'] / new['IPS_main_domain']
new['IPS_alt_frac'] = new['IPS_alt_boundary'] / new['IPS_alt_domain']
(new['IPS_alt_frac'] > new['IPS_main_frac']).mean(), new[['IPS_main_frac', 'IPS_alt_frac']].mean()

### ED10M: boundary-frequency distributions at differential bins (as in 11_check_diff_tad_boundaries)

In [ ]:
tad_dist = new.melt(value_vars=['IPS_main_frac', 'IPS_alt_frac'])
tad_dist.to_csv(f'{outdir}/ips_subtypes_tad_dist.csv', sep='\t')

### write tested / differential / non-differential bins (Table S11) and GREAT input

In [ ]:
tested.to_csv(f'{outdir}/ips_cluster_tested_bins.csv', sep='\t')
tested[tested['differential']].to_csv(f'{outdir}/ips_cluster_diff_bins.csv', sep='\t')
tested[~tested['differential']].to_csv(f'{outdir}/ips_cluster_nondiff_bins.csv', sep='\t')
tested.loc[tested['differential'], ['chrom', 'start', 'end']].to_csv(f'{beddir}/ips_cluster_diff_bins.bed', sep='\t', header=None, index=None)

In [ ]:
!date